# IPL Data Analysis using Python

## Project Overview

In this project, I analyzed IPL match and ball-by-ball datasets using Python and Pandas. The goal of this analysis is to understand team performance, player performance, toss impact, batting trends, and match-winning strategies through Exploratory Data Analysis (EDA).

The analysis is based on two datasets:
- **matches.csv** – Contains match-level information.
- **deliveries.csv** – Contains ball-by-ball data for every IPL match.

**Dataset Coverage:** IPL seasons from **2008 to 2024**. All insights in this notebook are based on the available data up to the end of the **2024 IPL season**

In [ ]:
import pandas as pd
import numpy as np

## Data Loading

In [153]:
matches = pd.read_csv("../data/matches.csv")
deliveries = pd.read_csv("../data/deliveries.csv")

print("Matches Shape:", matches.shape)
print("Deliveries Shape:", deliveries.shape)

Matches Shape: (1095, 20)
Deliveries Shape: (260920, 17)


In [ ]:
matches.head()

In [7]:
matches.columns

Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2'],
      dtype='object')

 *Section1 :Team performace (Most successfull teams)*
 # IPL Data Analysis using Python

## Section 1: Team Performance (Most Successful Teams)

### Business Question

Which teams have won the highest number of matches in IPL history?

In [ ]:
matches['winner'].value_counts().head(10)

In [ ]:
import matplotlib.pyplot as plt
matches['winner'].value_counts().head(10).plot(kind ='bar')
plt.title("Top 10 IPL Teams by Wins")
plt.xlabel("Teams")
plt.ylabel("Wins")
plt.show()

### Observation

Mumbai Indians and Chennai Super Kings are among the most successful teams in IPL history based on total match wins.

# Section 2: Toss Analysis

### Business Question

How often does the toss-winning team win the match?

In [ ]:
toss_match_winner = matches[matches['toss_winner']== matches['winner']]
percentage = (len(toss_match_winner)/len(matches))*100
print(f"Toss winner also won the match:{percentage:.2f}%")


### Observation

Winning the toss provides some advantage, but it does not guarantee a match win. Many matches are still won by the team that loses the toss.

# Section 3: Player of the Match Awards

### Business Question

Which players have won the Player of the Match award most frequently?

In [ ]:
matches['player_of_match'].value_counts().head(10)

*AB de Villiers has won the most player of the match award*

# Section 4: Venue Analysis

### Business Question

Which venues have hosted the most IPL matches?


In [142]:
match_runs = deliveries.groupby('match_id')['total_runs'].sum().reset_index()
match_runs.head()

,match_id,total_runs
0,335982,304
1,335983,447
2,335984,261
3,335985,331
4,335986,222


In [143]:
venue_runs = pd.merge(match_runs,
                     matches[['id','venue']],
                     left_on='match_id',
                     right_on = 'id')
venue_runs.head()

,match_id,total_runs,id,venue
0,335982,304,335982,M Chinnaswamy Stadium
1,335983,447,335983,"Punjab Cricket Association Stadium, Mohali"
2,335984,261,335984,Feroz Shah Kotla
3,335985,331,335985,Wankhede Stadium
4,335986,222,335986,Eden Gardens


In [ ]:
venue_summary = venue_runs.groupby('venue')['total_runs'].agg(
    avg_score = 'mean',
    matches = 'count'
)
venue_summary.sort_values('avg_score',ascending = False).head(10)

### Observation

Some stadiums have hosted significantly more IPL matches than others, mainly because they are regular home grounds for IPL teams.

# Section 5: Chasing vs Defending

### Business Question

Are more IPL matches won while chasing or while defending a target?

In [147]:
result_counts = matches['result'].value_counts()
chasing_wins = result_counts['wickets']
defending_wins = result_counts['runs']
total_results = chasing_wins + defending_wins
chasing_pct = (chasing_wins/total_results)*100
defending_pct = (defending_wins/total_results)*100
print(f"Chasing wins percentage: {chasing_pct:.2f}%")
print(f"Defending wins percentage: {defending_pct:.2f}%")

Chasing wins percentage: 53.72%
Defending wins percentage: 46.28%


In [148]:
chasing_matches = matches[matches['result']=='wickets']

In [ ]:
chasing_matches['winner'].value_counts().head(10)

In [ ]:
team1_matches = matches['team1']
team2_matches = matches['team2']
all_teams = pd.concat([team1_matches,team2_matches])
total_matches_played = all_teams.value_counts()
total_matches_played.sort_values(ascending=False)

### Observation

The comparison shows whether teams have been more successful while chasing or defending, giving an overall view of IPL match trends.

# Section 6: Toss Conversion Rate

### Business Question

How frequently does the toss-winning team convert the toss advantage into a match win?

In [ ]:
matches[matches['toss_winner'] == matches['winner']]['winner'].value_counts()

In [56]:
toss_wins = matches['toss_winner'].value_counts()

toss_match_wins = matches[
    matches['toss_winner'] == matches['winner']
]['winner'].value_counts()

print(toss_wins.head())
print()
print(toss_match_wins.head())

toss_winner
Mumbai Indians                 143
Kolkata Knight Riders          122
Chennai Super Kings            122
Rajasthan Royals               120
Royal Challengers Bangalore    113
Name: count, dtype: int64

winner
Mumbai Indians                 78
Chennai Super Kings            75
Kolkata Knight Riders          68
Rajasthan Royals               60
Royal Challengers Bangalore    57
Name: count, dtype: int64


In [ ]:
toss_analysis = pd.DataFrame({
    'toss_wins': toss_wins,
    'toss_match_wins': toss_match_wins
})

toss_analysis['toss_conversion_rate'] = (
    toss_analysis['toss_match_wins']
    /
    toss_analysis['toss_wins']
) * 100

toss_analysis.sort_values(
    'toss_conversion_rate',
    ascending=False
).head(10)

### Observation

The toss conversion rate shows how often winning the toss leads to winning the match, helping measure the actual impact of the toss.

# Section 7: Top Run Scorers

### Business Question

Which players have scored the most runs in IPL history?

In [ ]:
deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10)

*Virat Kohli emerged as the highest run scorer in IPL history, highlighting his consistency and ability to perform across multiple seasons. Other notable performers include Shikhar Dhawan, Rohit Sharma, David Warner, and Suresh Raina.*

### Observation

The top run scorers have maintained consistent batting performances over multiple IPL seasons.

# Section 8: Top Wicket Takers

### Business Question

Which bowlers have taken the most wickets in IPL history?

In [ ]:
deliveries['dismissal_kind'].value_counts()

In [ ]:
valid_wickets = deliveries[
    deliveries['dismissal_kind'].isin(['bowled','caught','lbw','stumped','caught and bowled','hit wicket'])
]
valid_wickets.groupby('bowler')['dismissal_kind'].count().sort_values(ascending=False).head(10)

*Yuzvendra Chahal emerged as the highest wicket-taker in IPL history with 205 wickets. Other consistent performers include Piyush Chawla, Dwayne Bravo, Bhuvneshwar Kumar, Sunil Narine, and Ravichandran Ashwin.*

### Observation

The leading wicket-takers have consistently performed with the ball and played an important role in their teams' success.

# Section 9: Highest Successful Run Chase

### Business Question

What is the highest target successfully chased in IPL history?

In [ ]:
chasing_matches = matches[matches['result'] == 'wickets']

chasing_matches.nlargest(1, 'target_runs')

*Punjab Kings completed the highest successful run chase in IPL history by chasing a target of 262 runs against Kolkata Knight Riders in the 2024 season.*

### Observation

This match represents one of the greatest successful run chases in IPL history, highlighting the strength of aggressive batting under pressure.

# Section 10: Lowest Successfully Defended Total (Normal Matches Only)

### Business Question

What is the lowest first innings total that was successfully defended in a full 20-over IPL match?

In [81]:
first_innings_score = deliveries[deliveries['inning']==1].groupby('match_id')['total_runs'].sum().reset_index()
first_innings_score = first_innings_score.rename(
    columns={'total_runs': 'first_innings_score'}
)
first_innings_score.head()

,match_id,first_innings_score
0,335982,222
1,335983,240
2,335984,129
3,335985,165
4,335986,110


In [94]:
merged = pd.merge(first_innings_score,
                  matches,
                  left_on = 'match_id',
                  right_on = 'id')

merged.head()

,match_id,first_innings_score,id,season,city,date,match_type,player_of_match,venue,team1,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,222,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,240,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,...,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,129,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,...,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,165,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,...,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,110,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,...,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [ ]:
defended_matches = merged[(merged['result']=='runs') & (merged['method'].isna())].nsmallest(5,'first_innings_score').sort_values('first_innings_score',ascending = True)
defended_matches.head()

### Observation

Even low totals can be defended when bowlers perform exceptionally well. D/L affected matches were excluded to ensure a fair comparison.

# Section 11: Best Chasing Team

### Business Question

Which team has the highest success rate while chasing targets? (Minimum 30 chase attempts)


In [110]:
matches['toss_loser']= np.where(
    matches['toss_winner']== matches['team1'],
    matches['team2'],
    matches['team1']
    
)
matches['chasing_team'] = np.where(
    matches['toss_decision']== 'field',
    matches['toss_winner'],
    matches['toss_loser']
)


In [ ]:
matches[['team1',
         'team2',
         'toss_winner',
         'toss_loser',
         'toss_decision',
         'chasing_team']].head(10)

In [ ]:
filtered = matches[
    matches['winner']== matches['chasing_team']]
filtered['winner'].value_counts()

In [ ]:
chasing_analysis = pd.DataFrame({
    'chase_attempts': matches['chasing_team'].value_counts(),
    'chase_wins': filtered['winner'].value_counts()
})
chasing_analysis['chase_success_rate'] = (
    chasing_analysis['chase_wins']/chasing_analysis['chase_attempts'])*100
chasing_analysis[chasing_analysis['chase_attempts']>30].sort_values('chase_success_rate',ascending = False).head(10)

### Observation

This analysis compares teams based on their chasing success rate instead of total wins, making the comparison more meaningful.

# Section 12: Best Defending Team

### Business Question

Which team has the highest success rate while defending a target? (Minimum 30 defending attempts)

In [121]:
matches['defending_team']= np.where(
    matches['toss_decision']== 'bat',
    matches['toss_winner'],
    matches['toss_loser']
)

In [125]:
defending_filter =matches[
    matches['winner'] == matches['defending_team']]


In [ ]:
defending_analysis = pd.DataFrame({
    'defending_attempts': matches['defending_team'].value_counts(),
    'defending_wins': defending_filter['winner'].value_counts()
})
defending_analysis['defending_success_rate']= (
    defending_analysis['defending_wins']/defending_analysis['defending_attempts'])*100
defending_analysis[defending_analysis['defending_attempts']>30].sort_values('defending_success_rate',ascending = False).head(10)

### Observation

This analysis shows which teams have been most successful in defending totals after batting first.

# Section 13: Highest Team Totals

### Business Question

What are the highest team totals scored in a single IPL innings?

In [ ]:
team_totals = deliveries.groupby(
    ['match_id', 'inning', 'batting_team']
)['total_runs'].sum().reset_index()
team_totals = team_totals.rename(columns={'total_runs': 'team_total'})
team_totals.nlargest(10,'team_total')


### Observation

The highest team totals highlight some of the most explosive batting performances in IPL history.

# Section 14: Teams with Most 200+ Totals

### Business Question

Which teams have scored 200 or more runs most frequently in IPL history?

In [ ]:
team_totals[team_totals['team_total']>=200]['batting_team'].value_counts().head(10)


### Observation

Teams with the highest number of 200+ totals have consistently displayed aggressive and high-scoring batting performances.

## Conclusion

This project analyzed IPL data from 2008 to 2024 using Python and Pandas.

Key findings include:
- Mumbai Indians and Chennai Super Kings have been the most successful teams.
- A toss win does not always result in a match win.
- Some teams perform significantly better while chasing, while others are stronger at defending.
- A few players have consistently dominated batting and bowling records.
- Recent IPL seasons have seen a rise in high-scoring matches, with several teams crossing the 200-run mark multiple times.

## Note

This analysis is based on IPL data available up to the **2024 season**. Any matches played after the 2024 season are not included in this project.